# AI-assisted workflow

This notebook repeats the workflow of this week’s lesson, but this time an
AI assistant helps us write the code. The division of labour is simple:

- **You decide the GIS part**: which data, which coordinate reference system,
  which spatial predicate, which classes. You are the GIS expert, and these
  decisions go *into* your prompt.
- **The AI assistant helps with the Python part**: boilerplate, syntax you
  don’t remember, restructuring code, explaining error messages.

The responses shown below are *examples*. If you run the same prompts yourself,
you will get different code, and that is fine. What matters is that you can
read the code, run it, and check the result.

This week, we use an AI assistant to scale up the raster processing steps of
the lesson: instead of handling one elevation tile at a time, we process all
four tiles in a loop.

> **Habit of the week: scale up carefully, and document**
>
> When code runs on many files, mistakes are repeated many times, and they are
> harder to spot. Before running a batch process, test it on *one* file and
> check that result in detail. And once you rely on AI-assisted code in your
> work (for instance, in the final assignment), document what you asked and
> how you checked it, so that others (and future you) can follow your work.

## Your decisions

- Input: the four elevation tiles `L4133A.tif` to `L4133D.tif` (EPSG:3067, 2 m resolution).
- The *NoData* value (`-9999`) is excluded from all calculations.
- Elevation classes, as in the lesson: **low** (up to 10 m), **mid** (10–20 m), **high** (above 20 m), stored as `1`, `2`, `3`; *NoData* is stored as `0`.
- The classified tiles are saved as compressed GeoTIFFs in `data/classified/`.

In [ ]:
import pathlib
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"

## Step 1: summary statistics for all tiles

**Prompt:**

```text
I use rioxarray. Write a function summarise_raster(path) that opens a
single-band GeoTIFF with its NoData value masked, and returns a dictionary with
the file name, min, max and mean value, and the share of valid (not NoData)
cells. Then apply it to all files matching "L4133*.tif" in DATA_DIRECTORY (a
pathlib.Path) and collect the results in a pandas DataFrame.
```

**Example response** (yours may differ):

In [ ]:
import pandas
import rioxarray as rxr


def summarise_raster(path):
    """Return summary statistics of a single-band raster, ignoring NoData."""
    raster = rxr.open_rasterio(path, masked=True)
    return {
        "file": path.name,
        "min": float(raster.min()),
        "max": float(raster.max()),
        "mean": float(raster.mean()),
        "valid_share": float(raster.notnull().mean()),
    }


raster_files = sorted(DATA_DIRECTORY.glob("L4133*.tif"))

summary = pandas.DataFrame([summarise_raster(path) for path in raster_files]).set_index("file")
summary

**Before you trust it**: `masked=True` replaces the NoData value with `NaN`,
and `NaN` values are ignored by `min()`, `max()` and `mean()`. Without it,
the value `-9999` would end up as the minimum, and pull down the mean.

(The `valid_share` column shows that these four tiles happen to have no
NoData cells at all. In the lesson, NoData cells appeared after clipping the
mosaic. The function works for both cases.)

Let’s test this on one file, in the same way we did it by hand in the lesson:

In [ ]:
raster = rxr.open_rasterio(raster_files[0])
nodata_value = raster.rio.nodata
valid_cells = raster.where(raster != nodata_value)

assert summary.loc[raster_files[0].name, "min"] == float(valid_cells.min())
assert summary.loc[raster_files[0].name, "max"] == float(valid_cells.max())
assert (summary["min"] > nodata_value).all(), "NoData values were included"

print(f"NoData value {nodata_value} is excluded; results match the manual calculation.")

## Step 2: classify all tiles and save them

**Prompt:**

```text
Write a function classify_raster(path, bins, output_directory) using rioxarray
and numpy. It opens a single-band GeoTIFF and classifies the values with
numpy.digitize into classes 1, 2, 3, ... using the class upper
bounds in `bins` (upper bounds inclusive). NoData cells get the value 0. Save
the result as uint8 GeoTIFF with NoData = 0 and deflate compression, with the
same file name, in output_directory. Return the output path.
Then apply it to a list of raster files.
```

**Example response** (yours may differ):

In [ ]:
import numpy as np

OUTPUT_DIRECTORY = DATA_DIRECTORY / "classified"
OUTPUT_DIRECTORY.mkdir(exist_ok=True)

CLASS_UPPER_BOUNDS = [10, 20]  # low: <= 10 m, mid: <= 20 m, high: > 20 m


def classify_raster(path, bins, output_directory):
    """
    Classify a single-band raster into classes 1, 2, 3, ... and save it.

    Class upper bounds (`bins`) are inclusive; NoData cells get the value 0.
    """
    raster = rxr.open_rasterio(path)
    nodata_value = raster.rio.nodata

    classes = np.digitize(raster.values, bins, right=True) + 1
    classes = np.where(raster.values == nodata_value, 0, classes).astype("uint8")

    classified = raster.copy(data=classes)
    classified = classified.rio.write_nodata(0)

    output_path = output_directory / path.name
    classified.rio.to_raster(output_path, compress="deflate")
    return output_path

**Before you run it on all files**, read it through:

- `numpy.digitize(..., right=True)` returns `0` for values up to 10, `1` for
  values up to 20, and `2` for everything above. Adding `1` gives our class
  values 1, 2, 3.
- NoData cells (`-9999`) would also get a class from `digitize()` (class 1,
  ‘low’!), so `numpy.where()` sets them to `0` afterwards, the same way we
  handled NoData values in the lesson.
- `raster.copy(data=...)` keeps the coordinates and the CRS of the input
  raster, `write_nodata(0)` records the new NoData value.

Following this week’s habit, we first test it on **one** file, and check the
result in detail:

In [ ]:
test_output = classify_raster(raster_files[0], CLASS_UPPER_BOUNDS, OUTPUT_DIRECTORY)

original = rxr.open_rasterio(raster_files[0])
classified = rxr.open_rasterio(test_output)

# same grid and CRS
assert classified.rio.crs == original.rio.crs
assert classified.shape == original.shape
assert classified.rio.transform() == original.rio.transform()

# only the expected values, NoData stays NoData
assert set(np.unique(classified.values)) <= {0, 1, 2, 3}
assert classified.rio.nodata == 0
assert ((classified.values == 0) == (original.values == original.rio.nodata)).all()

# class boundaries: exactly 10 m is ‘low’, just above is ‘mid’
values, classes = original.values.ravel(), classified.values.ravel()
values_valid = values != original.rio.nodata
assert (classes[values_valid & (values <= 10)] == 1).all()
assert (classes[(values > 10) & (values <= 20)] == 2).all()
assert (classes[values > 20] == 3).all()

# close the file again, so that it can be overwritten in the next step
classified.close()

print("The test file was classified correctly.")

Great! Now we can safely run it for all tiles:

In [ ]:
output_files = [
    classify_raster(path, CLASS_UPPER_BOUNDS, OUTPUT_DIRECTORY)
    for path in raster_files
]
[path.name for path in output_files]

In [ ]:
import matplotlib.pyplot as plt
from rioxarray.merge import merge_arrays

mosaic = merge_arrays([rxr.open_rasterio(path, masked=True) for path in output_files])

ax = mosaic.squeeze().plot(
    cmap=plt.matplotlib.colors.ListedColormap(["blue", "green", "brown"]),
    levels=[0.5, 1.5, 2.5, 3.5],
    figsize=(8, 7)
)
ax.colorbar.set_ticks([1, 2, 3])
ax.colorbar.set_ticklabels(["Low", "Mid", "High"])
plt.title("Classified elevation, all four tiles")
plt.show()

## AI-use log

Whenever you use an AI assistant in this course (in exercises, only where a
task is marked **AI-LLM OK**, see the [course guidelines on AI
tools](https://autogis-site.readthedocs.io/en/latest/course-info/ai-tools.html)),
keep a short log of what you did. For this notebook, it could look like this:

| Step | What I asked | What I changed | How I checked it |
|------|--------------|----------------|------------------|
| 1 | A function for summary statistics of a raster, applied to all tiles | Nothing | Compared one tile with the manual calculation from the lesson, checked that NoData is excluded |
| 2 | A function that classifies a raster with my class bounds and saves it | Nothing | Tested on one file first: grid, CRS, class values, NoData, class boundaries; then ran it for all tiles and looked at the mosaic |

> **Final assignment**
>
> If you use an AI assistant for your final assignment (see the [course
> guidelines on AI
> tools](https://autogis-site.readthedocs.io/en/latest/course-info/ai-tools.html)),
> include a log like this one in your repository.